# Tracing an Ejentum cognitive harness agent loop in Langfuse

This cookbook shows how to instrument an agent that calls the [Ejentum cognitive harness](https://ejentum.com) REST API between an LLM call and its response, so each turn shows up in Langfuse as a nested trace with the harness retrieval, the model call, and their token usage.

The pattern is reusable. Any in-loop REST call to a third-party tool can be wrapped the same way; Ejentum is the worked example because the agent calls one endpoint with a `mode` argument, which keeps the cookbook short.

## What you'll set up

1. The Langfuse v3 client and a project to send traces to.
2. The `from langfuse.openai import openai` drop-in so OpenAI calls are auto-instrumented.
3. An `@observe`-decorated function around each Ejentum REST call so the harness retrieval appears as a child span, with the `mode` and a scaffold length attached via `update_current_span(metadata=...)`.
4. A two-turn agent that calls `harness_reasoning` then `harness_anti_deception` and answers a reasoning-heavy task each turn.

Open the trace in your Langfuse project to inspect the tree per turn.

## 1. Install

In [ ]:
%pip install -q "langfuse>=3.0" openai requests

## 2. Configure Langfuse

Set `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_HOST` from your Langfuse project. The host defaults to `https://cloud.langfuse.com`; self-hosted users should set this to their instance URL.

In [ ]:
import os

# Replace with your project's keys, or load from a .env file.
os.environ.setdefault("LANGFUSE_HOST", "https://cloud.langfuse.com")
assert os.environ.get("LANGFUSE_PUBLIC_KEY"), "LANGFUSE_PUBLIC_KEY is not set."
assert os.environ.get("LANGFUSE_SECRET_KEY"), "LANGFUSE_SECRET_KEY is not set."
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."
assert os.environ.get("EJENTUM_API_KEY"), "EJENTUM_API_KEY is not set. Get one at https://ejentum.com/dashboard"

from langfuse import get_client

langfuse = get_client()
assert langfuse.auth_check(), "Langfuse auth check failed; verify your keys."
print("Langfuse client authenticated. Project:", os.environ["LANGFUSE_HOST"])

## 3. Auto-instrument OpenAI and wrap the Ejentum REST call

Langfuse ships a drop-in for OpenAI (`from langfuse.openai import openai`). It captures the request, response, model, and token usage automatically. For Ejentum we add a thin `@observe`-decorated wrapper so each harness retrieval shows up as a child span in the same trace, with the `mode` and scaffold length surfaced via `langfuse.update_current_span(metadata=...)`.

In [ ]:
import requests
from langfuse import observe
from langfuse.openai import openai

EJENTUM_URL = "https://ejentum-main-ab125c3.zuplo.app/logicv1/"


@observe(name="ejentum.call")
def call_ejentum(query: str, mode: str = "reasoning") -> str:
    """Fetch a cognitive scaffold from the Ejentum REST gateway.

    Emits a Langfuse span with input, output, and metadata
    (mode, query_length, scaffold_length).
    """
    r = requests.post(
        EJENTUM_URL,
        headers={
            "Authorization": f"Bearer {os.environ['EJENTUM_API_KEY']}",
            "Content-Type": "application/json",
        },
        json={"query": query, "mode": mode},
        timeout=10,
    )
    r.raise_for_status()
    payload = r.json()
    scaffold = payload[0].get(mode, "") if isinstance(payload, list) else ""
    langfuse.update_current_span(
        metadata={
            "mode": mode,
            "query_length": len(query),
            "scaffold_length": len(scaffold),
        },
    )
    return scaffold

## 4. Run a two-turn agent and view the trace

Each turn does the same shape:

1. `call_ejentum(...)` (Langfuse span with mode + scaffold length).
2. `openai.chat.completions.create(...)` from the auto-instrumented client.

Both spans sit inside a single trace per turn, so you can drill from the trace summary into the harness retrieval, the system message it produced, and the model call that consumed it.

In [ ]:
@observe(name="harness_augmented_turn")
def harness_augmented_turn(task: str, mode: str = "reasoning") -> str:
    scaffold = call_ejentum(task, mode=mode)
    completion = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Apply the cognitive scaffold below before answering.\n\n"
                    f"[SCAFFOLD]\n{scaffold}\n[END]"
                ),
            },
            {"role": "user", "content": task},
        ],
        temperature=0,
    )
    return completion.choices[0].message.content or ""


print(harness_augmented_turn(
    "We have 50M users and want to add a NOT NULL column. "
    "Walk through the trade-offs and recommend a migration path.",
    mode="reasoning",
))
print("---")
print(harness_augmented_turn(
    "A user insists their DB is the bottleneck and refuses other diagnostics. "
    "Walk through whether the framing is sound before recommending.",
    mode="anti-deception",
))

# Flush so the traces show up immediately rather than on process exit.
langfuse.flush()

## What to look at in Langfuse

Open your Langfuse project. For each turn you should see one trace with two child spans:

- `ejentum.call`: the harness retrieval. Open the metadata panel to confirm `mode` and `scaffold_length`.
- `openai.chat.completions.create` (from `langfuse.openai`): the model call. Token usage, model name, prompt, and response are captured automatically.

Useful filters in the Traces view:

- `metadata.mode = "anti-deception"` to see only deception-resistance turns.
- `metadata.scaffold_length > 0` to confirm every harness call returned a non-empty scaffold.

## Adapting the pattern

Any in-loop REST call to a third-party tool can be instrumented the same way. The four steps in this cookbook (drop-in OpenAI client + `@observe`-decorated REST wrapper + metadata via `update_current_span` + scaffold-aware system message) generalise to harness calls in any agent loop. For larger agent frameworks, swap step 4 for the framework's native control flow; the span shape stays the same.